In [5]:
import sys
sys.path.append("/Users/monikagadage/Documents/java-interview-coach")

In [7]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class InterviewState(TypedDict):
    topic: str
    current_question: str
    user_answer: str
    feedback: str
    score: int
    total_questions: int
    needs_hint: bool
    hint: str
    weak_topics: list[str]
    next_action: str
    messages: Annotated[list, add_messages]

In [11]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END


In [12]:
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [13]:
question_prompt = ChatPromptTemplate.from_template("""
You are a Java technical interviewer.
Generate ONE clear interview question about: {topic}
Just the question, nothing else.
""")

eval_prompt = ChatPromptTemplate.from_template("""
You are a Java technical interviewer evaluating an answer.

Question: {question}
Candidate's Answer: {answer}

Respond with:
1. CORRECT or INCORRECT
2. Brief feedback (2-3 sentences)
3. Ideal answer in simple terms

Start your response with either CORRECT or INCORRECT on the first line.
""")

hint_prompt = ChatPromptTemplate.from_template("""
You are a helpful Java tutor.
Give a short hint (2-3 sentences) for this question without giving away the answer.
Question: {question}
""")


In [14]:
question_chain = question_prompt | llm
eval_chain = eval_prompt | llm
hint_chain = hint_prompt | llm



In [16]:
def ask_question_node(state : InterviewState) -> dict:
    """Generates a new Interview Question"""
    response = question_chain.invoke({"topic" : state["topic"]})
    return{
        "current_question" : response.content,
        "total_questions" : state.get("total_questions", 0) + 1,
        "need_hints" : False,
        "hint" : ""
    }
    

In [27]:
def evaluate_node(state: InterviewState) -> dict:
    """Evaluates the user's answers"""
    response = eval_chain.invoke({
        "question": state["current_question"],
        "answer" : state["user_answer"]
    })
    
    feedback = response.content
    is_correct = feedback.strip().upper().startswith("CORRECT")
    
    score = state.get("score", 0)
    weak_topics = state.get("weak_topics" , [])
    
    if is_correct:
        score+=1
    else:
        weak_topics = weak_topics+ [state["topic"]]
        
    return {
        "feedback" : feedback,
        "score" : score,
        "weak_topics" : weak_topics
    }
    

In [24]:
def hint_node(state: InterviewState) -> dict:
    """Gives a hint for the current question"""
    response = hint_chain.invoke({
        "question": state["current_question"]
    })
    return {"hint": response.content}

In [19]:
def end_node(state: InterviewState) -> dict:
    """Shows final score"""
    return {"next_action": "end"}

In [22]:
def router(state: InterviewState) -> str:
    """Decides what to do next based on next_action"""
    action = state.get("next_action", "ask")

    if action == "end":
        return "end"
    elif action == "hint":
        return "hint"
    elif action == "evaluate":
        return "evaluate"
    else:
        return "ask"

In [21]:
def build_graph():
    graph = StateGraph(InterviewState)
    
    graph.add_node("ask" , ask_question_node)
    graph.add_node("evaluate", evaluate_node)
    graph.add_node("hint", hint_node)
    graph.add_node("end", end_node)
    
    graph.set_entry_point("ask")
    
    graph.add_edge("ask" , "evaluate")
    
    graph.add_conditional_edges(
        "evaluate",
        router,{
            "ask": "ask",
            "hint": "hint",
            "end": END
        }
    )
    graph.add_edge("hint", "evaluate")

    return graph.compile()


In [ ]:
from dotenv import load_dotenv
load_dotenv()

# Build the graph
graph = build_graph()

def run_interview():
    print("🎯 Java Interview Coach\n")
    topic = input("Enter a topic (e.g. OOP, Collections, JVM, Spring Boot): ")
    
    # Initial state
    state = {
        "topic": topic,
        "current_question": "",
        "user_answer": "",
        "feedback": "",
        "score": 0,
        "total_questions": 0,
        "needs_hint": False,
        "hint": "",
        "weak_topics": [],
        "next_action": "ask",
        "messages": []
    }

    for _ in range(5):  # 5 questions per session
        # Ask question
        state = graph.invoke(state)
        print(f"\n📌 Question {state['total_questions']}:\n{state['current_question']}\n")

        # Get user input
        user_input = input("Your answer (or type 'hint'): ")

        if user_input.lower() == "hint":
            state["needs_hint"] = True
            state["next_action"] = "hint"
            state["user_answer"] = ""
            state = graph.invoke(state)
            print(f"\n💡 Hint:\n{state['hint']}\n")
            user_input = input("Now your answer: ")

        # Evaluate
        state["user_answer"] = user_input
        state["next_action"] = "evaluate"
        state = graph.invoke(state)
        print(f"\n💬 Feedback:\n{state['feedback']}\n")
        print(f"⭐ Score: {state['score']}/{state['total_questions']}")

        # Ask to continue
        cont = input("\nNext question? (y/n): ")
        if cont.lower() != "y":
            break
        
        state["next_action"] = "ask"

    # Final summary
    print(f"\n🏁 Session Complete!")
    print(f"✅ Final Score: {state['score']}/{state['total_questions']}")
    if state["weak_topics"]:
        print(f"📚 Topics to review: {', '.join(set(state['weak_topics']))}")

run_interview()

🎯 Java Interview Coach

